In [1]:
"""
05_feature_engineering
"""

'\n'

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from utils.load_data_model import load_data_model

df = load_data_model()

In [4]:

import numpy as np
from utils.print_section import print_section

# ============================================================
# Paths
# ============================================================

OUTPUT_PATH = "../data/processed/model_features.csv"
LOG_PATH = "../logs/feature_engineering_log.txt"

print(df.shape)

# ============================================================
# Create financial ratios
# ============================================================

print_section("Create financial ratios")

df["profitability"] = (
    df["net_income"] / df["ta"]
).replace(
    [np.inf, -np.inf],
    np.nan
)

df["liquidity"] = (
    df["current_assets"] / df["st_debt"]
).replace(
    [np.inf, -np.inf],
    np.nan
)

df["solvency"] = (
    df["rub10_15"] / df["ta"]
).replace(
    [np.inf, -np.inf],
    np.nan
)

df["structure"] = (
    df["st_debt"] / df["total_debt"]
).replace(
    [np.inf, -np.inf],
    np.nan
)

df["log_age"] = np.log1p(
    df["age_in_years"]
)

RATIO_COLUMNS = [
    "profitability",
    "liquidity",
    "solvency",
    "structure",
    "log_age",
]

print(
    df[RATIO_COLUMNS]
    .describe()
)

# ============================================================
# Winsorize ratios
# ============================================================

print_section("Winsorize ratios")

winsor_log = {}

for col in [
    "profitability",
    "liquidity",
    "solvency",
    "structure",
]:

    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)

    lower_count = (
        df[col] < lower
    ).sum()

    upper_count = (
        df[col] > upper
    ).sum()

    df[col] = df[col].clip(
        lower=lower,
        upper=upper
    )

    winsor_log[col] = {
        "lower": lower,
        "upper": upper,
        "lower_count": lower_count,
        "upper_count": upper_count,
    }

    print(
        f"{col}: "
        f"[{lower:.4f}, {upper:.4f}] "
        f"({lower_count:,} lower, "
        f"{upper_count:,} upper)"
    )

# ============================================================
# Missing values
# ============================================================

print_section("Missing values")

print(
    df[RATIO_COLUMNS]
    .isna()
    .sum()
)

# ============================================================
# Save dataset
# ============================================================

print_section("Save feature dataset")

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Saved to: {OUTPUT_PATH}"
)

# ============================================================
# Write log
# ============================================================

print_section("Write log")

with open(LOG_PATH, "w") as f:

    f.write(
        "FEATURE ENGINEERING LOG\n"
    )

    f.write(
        "=" * 60 + "\n\n"
    )

    f.write(
        f"Rows: {len(df):,}\n"
    )

    f.write(
        f"Columns: {len(df.columns):,}\n\n"
    )

    f.write(
        "Created features\n"
    )

    f.write(
        "-" * 60 + "\n"
    )

    for col in RATIO_COLUMNS:

        f.write(
            f"{col}\n"
        )

    f.write("\n")

    f.write(
        "Winsorization (1st/99th percentile)\n"
    )

    f.write(
        "-" * 60 + "\n"
    )

    for col, stats in winsor_log.items():

        f.write(
            f"{col}: "
            f"[{stats['lower']:.4f}, "
            f"{stats['upper']:.4f}]\n"
        )

print(
    f"Log written to: {LOG_PATH}"
)

(981818, 91)

Create financial ratios
       profitability      liquidity      solvency      structure  \
count  976617.000000  971498.000000  9.793480e+05  975482.000000   
mean       -0.564690      20.389385 -4.025326e+01       0.740281   
std       849.016186    1809.452812  1.379330e+04       0.303567   
min   -508986.000000       0.000000 -8.037153e+06       0.000000   
25%        -0.012041       0.770704  1.172710e-01       0.500452   
50%         0.036035       1.496694  3.803197e-01       0.893156   
75%         0.122834       3.286082  6.807328e-01       1.000000   
max    584889.000000  801339.785235  2.338306e+01      10.334122   

             log_age  
count  981818.000000  
mean        2.443545  
std         0.745339  
min         0.000000  
25%         1.945910  
50%         2.484907  
75%         2.995732  
max         4.852030  

Winsorize ratios
profitability: [-2.5710, 0.7644] (9,767 lower, 9,767 upper)
liquidity: [0.0060, 100.6048] (9,715 lower, 9,715 upper)
solvenc